In [1]:
# ==========================================
# MICROSOFT PHI-4-MINI — LOAD MODEL
# ==========================================

import torch
import gc
import os
import time
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "microsoft/Phi-4-mini-instruct"

print("=" * 70)
print("LOADING MICROSOFT PHI-4-MINI")
print("=" * 70)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

print("Model loaded.")
print("Device:", model.device)

LOADING MICROSOFT PHI-4-MINI


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.
Device: cuda:0


In [5]:
# ==========================================
# PHI-4-MINI — FEW-SHOT SAMPLE 0 TEST
# ==========================================

FEW_SHOT_EXAMPLES = [
    {
        "original": "The spacecraft consists of two main elements: the NASA Cassini orbiter and the ESA Huygens probe.",
        "simplified": "The spacecraft has two main parts: the Cassini orbiter and the Huygens probe."
    },
    {
        "original": "Jeddah is the principal gateway to Mecca, Islam's holiest city.",
        "simplified": "Jeddah is the main entrance to Mecca, Islam's holiest city."
    },
    {
        "original": "His next work follows an especially eventful day in the life of a successful neurosurgeon.",
        "simplified": "His next work is about a very busy day in the life of a successful brain surgeon."
    }
]


def few_shot_prompt(sentence):

    examples_text = ""

    for example in FEW_SHOT_EXAMPLES:
        examples_text += (
            f"Original: {example['original']}\n"
            f"Simplified: {example['simplified']}\n\n"
        )

    return f"""Simplify the following sentences using clear and easy-to-understand language.

Preserve the original meaning.
Do not add explanations or commentary.
Return only the simplified sentence.

Here are some examples:

{examples_text}

Now simplify this sentence:

Original: {sentence}
Simplified:"""


# ==========================================
# SAMPLE 0
# ==========================================

example = asset_df.iloc[0]

source = example["original"]

prompt = few_shot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Phi's official chat-template approach
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

start_time = time.perf_counter()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = end_time - start_time

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[-1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

peak_vram = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)

print("=" * 70)
print("PHI-4-MINI — FEW-SHOT SAMPLE 0")
print("=" * 70)

print("\nORIGINAL:")
print(source)

print("\nPHI GENERATED:")
print(prediction)

print("\nInference:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")


# ==========================================
# METRICS
# ==========================================

test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():
    print(f"{metric}: {value:.4f}")

PHI-4-MINI — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

PHI GENERATED:
Marin is south, Mendocino is north, Lake is northeast, Napa is east, and Solano and Contra Costa are southeast.

Inference: 2.189 sec
Peak VRAM: 7.255 GB


NameError: name 'evaluate_one_sample' is not defined

In [4]:
# ==========================================
# RELOAD ASSET — 2359 SAMPLES
# ==========================================

from datasets import load_dataset
import pandas as pd

print("Loading ASSET...")

asset = load_dataset("facebook/asset")

asset_df = pd.concat(
    [
        pd.DataFrame(asset["validation"]),
        pd.DataFrame(asset["test"])
    ],
    ignore_index=True
)

print("=" * 70)
print("ASSET READY")
print("=" * 70)
print("Validation:", len(asset["validation"]))
print("Test      :", len(asset["test"]))
print("Total     :", len(asset_df))
print("Columns   :", asset_df.columns.tolist())

Loading ASSET...
ASSET READY
Validation: 2000
Test      : 359
Total     : 2359
Columns   : ['original', 'simplifications']


In [8]:
# ==========================================
# METRICS FUNCTION
# SAME PIPELINE USED FOR ALL MODELS
# ==========================================

!pip install -q evaluate textstat sacremoses sacrebleu

import evaluate
import textstat
import numpy as np

print("Loading metrics...")

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("All metrics loaded.")


def evaluate_one_sample(source, prediction, references):

    # Safety for empty output
    if prediction is None or str(prediction).strip() == "":
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }

    prediction = str(prediction).strip()

    # --------------------------------------
    # SARI
    # --------------------------------------

    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    sari = sari_result["sari"]

    # --------------------------------------
    # BLEU
    # --------------------------------------

    bleu_result = bleu_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    bleu = bleu_result["bleu"]

    # --------------------------------------
    # ROUGE-L
    # --------------------------------------

    rouge_result = rouge_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    rougeL = rouge_result["rougeL"]

    # --------------------------------------
    # BERTScore
    # --------------------------------------

    bert_result = bertscore_metric.compute(
        predictions=[prediction],
        references=[references[0]],
        lang="en"
    )

    bertscore_f1 = bert_result["f1"][0]

    # --------------------------------------
    # FKGL
    # --------------------------------------

    try:
        fkgl = textstat.flesch_kincaid_grade(prediction)
    except:
        fkgl = 0.0

    return {
        "sari": sari,
        "bleu": bleu,
        "rougeL": rougeL,
        "bertscore_f1": bertscore_f1,
        "fkgl": fkgl
    }


print("✅ evaluate_one_sample() is ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 6.9 MB/s eta 0:00:00
Loading metrics...


ImportError: To be able to use evaluate-metric/rouge, you need to install the following dependencies['rouge_score'] using 'pip install rouge_score' for instance'

In [9]:
!pip install -q rouge_score sacremoses

  Preparing metadata (setup.py) ... done


In [11]:
# ==========================================
# LOAD EVALUATION METRICS
# ==========================================

!pip install -q evaluate textstat sacremoses sacrebleu bert_score

import evaluate
import textstat
import numpy as np

print("Loading metrics...")

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("All metrics loaded successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.3 MB/s eta 0:00:00
Loading metrics...
All metrics loaded successfully.


In [12]:
test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():
    print(f"{metric}: {value:.4f}")

NameError: name 'evaluate_one_sample' is not defined

In [13]:
# ==========================================
# PHI — EVALUATION METRICS + FUNCTION
# ==========================================

import evaluate
import textstat
import numpy as np

print("Loading metrics...")

sari_metric = evaluate.load("sari")
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("All metrics loaded successfully.")


def evaluate_one_sample(source, prediction, references):

    # Handle empty generation
    if prediction is None or str(prediction).strip() == "":
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }

    prediction = str(prediction).strip()

    # ======================================
    # SARI
    # ======================================

    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # BLEU
    # ======================================

    bleu_result = bleu_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # ROUGE-L
    # ======================================

    rouge_result = rouge_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # BERTScore
    # ======================================

    bert_result = bertscore_metric.compute(
        predictions=[prediction],
        references=[references[0]],
        lang="en"
    )

    # ======================================
    # FKGL
    # ======================================

    try:
        fkgl = textstat.flesch_kincaid_grade(prediction)
    except Exception:
        fkgl = 0.0

    return {
        "sari": sari_result["sari"],
        "bleu": bleu_result["bleu"],
        "rougeL": rouge_result["rougeL"],
        "bertscore_f1": bert_result["f1"][0],
        "fkgl": fkgl
    }


print("✅ evaluate_one_sample() is ready.")

Loading metrics...
All metrics loaded successfully.
✅ evaluate_one_sample() is ready.


In [14]:
# ==========================================
# PHI-4-MINI — FEW-SHOT SAMPLE 0 METRICS
# ==========================================

test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("=" * 70)
print("PHI-4-MINI — FEW-SHOT SAMPLE 0")
print("=" * 70)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():
    print(f"{metric}: {value:.4f}")

print("\nInference:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PHI-4-MINI — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GENERATED:
Marin is south, Mendocino is north, Lake is northeast, Napa is east, and Solano and Contra Costa are southeast.

METRICS
sari: 38.5776
bleu: 0.3539
rougeL: 0.7568
bertscore_f1: 0.8948
fkgl: 10.4516

Inference: 2.189 sec
Peak VRAM: 7.255 GB


In [15]:
# ==========================================
# PHI-4-MINI — FEW-SHOT
# EXACT SAME METRIC PIPELINE AS GEMMA
# ALL ASSET REFERENCES USED
# ==========================================

def evaluate_one_sample(source, prediction, references):

    # --------------------------------------
    # EMPTY OUTPUT
    # --------------------------------------
    if prediction is None or str(prediction).strip() == "":
        return {
            "sari": 0.0,
            "bleu": 0.0,
            "rougeL": 0.0,
            "bertscore_f1": 0.0,
            "fkgl": 0.0
        }

    prediction = str(prediction).strip()

    # Make sure references are a list
    references = list(references)

    # ======================================
    # SARI — ALL REFERENCES
    # ======================================

    sari_result = sari_metric.compute(
        sources=[source],
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # BLEU — ALL REFERENCES
    # ======================================

    bleu_result = bleu_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # ROUGE-L
    # Use the same reference treatment
    # as your Gemma evaluation
    # ======================================

    rouge_result = rouge_metric.compute(
        predictions=[prediction],
        references=[references]
    )

    # ======================================
    # BERTSCORE
    # Score against all 10 references
    # ======================================

    bert_predictions = [prediction] * len(references)

    bert_result = bertscore_metric.compute(
        predictions=bert_predictions,
        references=references,
        lang="en"
    )

    # Average BERTScore across the 10 references
    bertscore_f1 = float(np.mean(bert_result["f1"]))

    # ======================================
    # FKGL
    # ======================================

    try:
        fkgl = textstat.flesch_kincaid_grade(prediction)
    except Exception:
        fkgl = 0.0

    return {
        "sari": float(sari_result["sari"]),
        "bleu": float(bleu_result["bleu"]),
        "rougeL": float(rouge_result["rougeL"]),
        "bertscore_f1": bertscore_f1,
        "fkgl": float(fkgl)
    }

print("✅ Corrected metric function ready.")

✅ Corrected metric function ready.


In [16]:
# ==========================================
# RE-SCORE CURRENT PHI SAMPLE
# ==========================================

test_scores = evaluate_one_sample(
    source,
    prediction,
    example["simplifications"]
)

print("=" * 70)
print("PHI-4-MINI — FEW-SHOT SAMPLE 0")
print("=" * 70)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

print("\nMETRICS")
print("=" * 50)

for metric, value in test_scores.items():
    print(f"{metric}: {value:.4f}")

print("\nInference:", round(inference_time, 3), "sec")
print("Peak VRAM:", round(peak_vram, 3), "GB")

print("\nNumber of references used:",
      len(example["simplifications"]))

PHI-4-MINI — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GENERATED:
Marin is south, Mendocino is north, Lake is northeast, Napa is east, and Solano and Contra Costa are southeast.

METRICS
sari: 38.5776
bleu: 0.3539
rougeL: 0.7568
bertscore_f1: 0.8962
fkgl: 10.4516

Inference: 2.189 sec
Peak VRAM: 7.255 GB

Number of references used: 10


In [18]:
# ==========================================
# PHI-4-MINI — FEW-SHOT TEXT GENERATION
# ==========================================

example = asset_df.iloc[0]

source = example["original"]

# Build the SAME few-shot prompt
prompt = few_shot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# Clear CUDA memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Tokenize using Phi-4 chat template
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# Generate
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Get ONLY generated text
generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[-1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

print("=" * 80)
print("PHI-4-MINI — FEW-SHOT TEXT GENERATION")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

PHI-4-MINI — FEW-SHOT TEXT GENERATION

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GENERATED:
Marin is south, Mendocino is north, Lake is northeast, Napa is east, and Solano and Contra Costa are southeast.


In [19]:
# ==========================================
# PHI-4-MINI — FEW-SHOT
# GENERATE + METRICS + RESOURCE USAGE
# ==========================================

example = asset_df.iloc[0]

source = example["original"]
references = example["simplifications"]

# ==========================================
# BUILD FEW-SHOT PROMPT
# ==========================================

prompt = few_shot_prompt(source)

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# ==========================================
# CLEAR GPU MEMORY
# ==========================================

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ==========================================
# TOKENIZE
# ==========================================

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# ==========================================
# GENERATION + TIME
# ==========================================

start_time = time.perf_counter()

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

inference_time = end_time - start_time

# ==========================================
# DECODE ONLY GENERATED TEXT
# ==========================================

generated_ids = output_ids[
    0,
    inputs["input_ids"].shape[-1]:
]

prediction = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
).strip()

# ==========================================
# PEAK VRAM
# ==========================================

peak_vram = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if torch.cuda.is_available()
    else None
)

# ==========================================
# METRICS — ALL 10 REFERENCES
# ==========================================

scores = evaluate_one_sample(
    source,
    prediction,
    references
)

# ==========================================
# FINAL OUTPUT
# ==========================================

print("=" * 80)
print("PHI-4-MINI — FEW-SHOT SAMPLE 0")
print("=" * 80)

print("\nORIGINAL:")
print(source)

print("\nGENERATED:")
print(prediction)

print("\nMETRICS")
print("-" * 50)

print(f"SARI          : {scores['sari']:.4f}")
print(f"BLEU          : {scores['bleu']:.4f}")
print(f"ROUGE-L       : {scores['rougeL']:.4f}")
print(f"BERTScore F1  : {scores['bertscore_f1']:.4f}")
print(f"FKGL          : {scores['fkgl']:.4f}")

print("\nRESOURCE")
print("-" * 50)

print(f"Inference     : {inference_time:.3f} sec")
print(f"Peak VRAM     : {peak_vram:.3f} GB")

print("\nReferences used:", len(references))

# ==========================================
# CLEANUP
# ==========================================

del inputs
del output_ids
del generated_ids

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

PHI-4-MINI — FEW-SHOT SAMPLE 0

ORIGINAL:
Adjacent counties are Marin (to the south), Mendocino (to the north), Lake (northeast), Napa (to the east), and Solano and Contra Costa (to the southeast).

GENERATED:
Marin is south, Mendocino is north, Lake is northeast, Napa is east, and Solano and Contra Costa are southeast.

METRICS
--------------------------------------------------
SARI          : 38.5776
BLEU          : 0.3539
ROUGE-L       : 0.7568
BERTScore F1  : 0.8962
FKGL          : 10.4516

RESOURCE
--------------------------------------------------
Inference     : 2.490 sec
Peak VRAM     : 8.278 GB

References used: 10


In [21]:
 # ==========================================
# PHI-4-MINI — FEW-SHOT
# GENERATE TEXT + SCORE EACH SAMPLE
# ASSET 2359
# ==========================================

FEW_SHOT_OUTPUT = "PHI_FEW_SHOT_ASSET_2359_FINAL.csv"
CHECKPOINT_EVERY = 25

if os.path.exists(FEW_SHOT_OUTPUT):

    fewshot_df = pd.read_csv(FEW_SHOT_OUTPUT)

    completed_ids = set(
        fewshot_df["sample_id"].astype(int)
    )

    print(f"🔄 Resuming: {len(completed_ids)}/2359 completed.")

else:

    fewshot_df = pd.DataFrame()
    completed_ids = set()

    print("🆕 Starting Phi Few-Shot from sample 1.")


for i in range(len(asset_df)):

    if i in completed_ids:
        continue

    example = asset_df.iloc[i]

    source = example["original"]
    references = example["simplifications"]

    # ======================================
    # BUILD FEW-SHOT PROMPT
    # ======================================

    prompt = few_shot_prompt(source)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    # ======================================
    # GPU RESET
    # ======================================

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    # ======================================
    # TOKENIZE
    # ======================================

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    # ======================================
    # GENERATE TEXT
    # ======================================

    start_time = time.perf_counter()

    try:

        with torch.no_grad():

            output_ids = model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_ids = output_ids[
            0,
            inputs["input_ids"].shape[-1]:
        ]

        prediction = tokenizer.decode(
            generated_ids,
            skip_special_tokens=True
        ).strip()

    except Exception as e:

        prediction = ""

        print(f"\n⚠️ ERROR on sample {i}: {e}")

    end_time = time.perf_counter()

    inference_time = end_time - start_time

    # ======================================
    # PEAK VRAM
    # ======================================

    peak_vram = (
        torch.cuda.max_memory_allocated() / (1024 ** 3)
        if torch.cuda.is_available()
        else None
    )

    # ======================================
    # SCORE
    # ======================================

    scores = evaluate_one_sample(
        source,
        prediction,
        references
    )

    # ======================================
    # PRINT ACTUAL TEXT
    # ======================================

    print("\n")
    print("=" * 90)
    print(f"SAMPLE {i + 1}/2359")
    print("=" * 90)

    print("\nORIGINAL:")
    print(source)

    print("\nPHI GENERATED:")
    print(prediction if prediction else "[EMPTY]")

    print("\nMETRICS:")
    print(f"SARI          : {scores['sari']:.4f}")
    print(f"BLEU          : {scores['bleu']:.4f}")
    print(f"ROUGE-L       : {scores['rougeL']:.4f}")
    print(f"BERTScore F1  : {scores['bertscore_f1']:.4f}")
    print(f"FKGL          : {scores['fkgl']:.4f}")

    print("\nRESOURCE:")
    print(f"Inference     : {inference_time:.3f} sec")
    print(f"Peak VRAM     : {peak_vram:.3f} GB")

    # ======================================
    # SAVE RESULT
    # ======================================

    result = {
        "sample_id": i,
        "model": "Phi-4-mini-instruct",
        "prompting": "few-shot",

        "original": source,
        "generated": prediction,
        "references": " ||| ".join(references),

        "sari": scores["sari"],
        "bleu": scores["bleu"],
        "rougeL": scores["rougeL"],
        "bertscore_f1": scores["bertscore_f1"],
        "fkgl": scores["fkgl"],

        "inference_time_sec": inference_time,
        "peak_vram_gb": peak_vram
    }

    fewshot_df = pd.concat(
        [
            fewshot_df,
            pd.DataFrame([result])
        ],
        ignore_index=True
    )

    completed_ids.add(i)

    # ======================================
    # CHECKPOINT EVERY 25
    # ======================================

    if (
        len(completed_ids) % CHECKPOINT_EVERY == 0
        or i == len(asset_df) - 1
    ):

        fewshot_df.to_csv(
            FEW_SHOT_OUTPUT,
            index=False
        )

        print(
            f"\n💾 CHECKPOINT SAVED: "
            f"{len(completed_ids)}/2359"
        )

    # ======================================
    # CLEANUP
    # ======================================

    del inputs
    del output_ids
    del generated_ids

    if i % 10 == 0:

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ==========================================
# FINAL SAVE
# ==========================================

fewshot_df.to_csv(
    FEW_SHOT_OUTPUT,
    index=False
)

print("\n")
print("=" * 90)
print("✅ PHI-4-MINI FEW-SHOT COMPLETE")
print("=" * 90)
print(f"Completed: {len(fewshot_df)}/2359")
print(f"CSV      : {FEW_SHOT_OUTPUT}")

Streaming output truncated to the last 5000 lines.
Peak VRAM     : 8.275 GB


SAMPLE 2134/2359

ORIGINAL:
Their participation is limited until they reach legal adulthood.

PHI GENERATED:
They can only join until they become adults.

METRICS:
SARI          : 42.0956
BLEU          : 0.4671
ROUGE-L       : 0.7059
BERTScore F1  : 0.9207
FKGL          : 5.2300

RESOURCE:
Inference     : 0.464 sec
Peak VRAM     : 8.267 GB


SAMPLE 2135/2359

ORIGINAL:
Development Stable releases are rare, but there are often Subversion snapshots which are stable enough to use.

PHI GENERATED:
Stable releases are rare, but there are often stable snapshots.

METRICS:
SARI          : 33.0484
BLEU          : 0.7633
ROUGE-L       : 0.7200
BERTScore F1  : 0.9184
FKGL          : 7.1900

RESOURCE:
Inference     : 0.597 sec
Peak VRAM     : 8.272 GB


SAMPLE 2136/2359

ORIGINAL:
Finally in 1482 the Order dispatched him to Florence, the ‘ city of his destiny ’.

PHI GENERATED:
Finally in 1482, the Order sent him to Flo

In [22]:
import pandas as pd

# Load CSV
df = pd.read_csv("PHI_FEW_SHOT_ASSET_2359_FINAL.csv")

# Statistics for all numeric columns
stats = df.select_dtypes(include="number").agg(
    ["mean", "median", "min", "max"]
).T

print(stats)

                           mean       median        min          max
sample_id           1179.000000  1179.000000   0.000000  2358.000000
sari                  44.331509    44.239351  18.409171    80.275632
bleu                   0.672567     0.699206   0.000000     1.000000
rougeL                 0.779513     0.790698   0.160000     1.000000
bertscore_f1           0.947813     0.949177   0.793571     0.993692
fkgl                   8.654094     8.541538  -1.450000    25.473333
inference_time_sec     1.047782     0.977001   0.283395     4.331929
peak_vram_gb           8.273581     8.272410   8.264496     8.308686


In [23]:
stats.to_csv("PHI_FEW_SHOT_ASSET_2359_STATS.csv")

print(stats)

                           mean       median        min          max
sample_id           1179.000000  1179.000000   0.000000  2358.000000
sari                  44.331509    44.239351  18.409171    80.275632
bleu                   0.672567     0.699206   0.000000     1.000000
rougeL                 0.779513     0.790698   0.160000     1.000000
bertscore_f1           0.947813     0.949177   0.793571     0.993692
fkgl                   8.654094     8.541538  -1.450000    25.473333
inference_time_sec     1.047782     0.977001   0.283395     4.331929
peak_vram_gb           8.273581     8.272410   8.264496     8.308686
